# Análisis de video y estado del tráfico

Usá esta notebook para aplicar un bundle validado a un video, anotar vehículos y obtener estados por minuto. PostgreSQL y la revisión humana sólo se activan mediante un preset explícito.

| Preset | Resultado | Efecto externo |
|---|---|---|
| `InferencePreset.PILOT_OFFLINE` | Video, telemetría y dashboard | Ninguno |
| `InferencePreset.PILOT_HITL` | Resultado + ZIP local de revisión | Publicación Drive separada y explícita |
| `InferencePreset.PERSISTED_INFERENCE` | Resultado persistido | PostgreSQL `inference` |
| `InferencePreset.PERSISTED_HITL` | Persistencia + ZIP local de revisión | PostgreSQL; Drive sólo al publicar |
| `InferencePreset.EXPERIMENTAL_OFFLINE` | Candidato sin aprobar | Siempre offline |
| `InferencePreset.CUSTOM` | Un preset base con cambios explícitos | Depende de la configuración |

**Inicio rápido recomendado:** conservá `InferencePreset.PILOT_OFFLINE`. Autorizar un bundle permite probarlo; no lo promociona. `Accident` nunca es una salida automática.

<details>
<summary><strong>Personalización tipada</strong></summary>

Seleccioná `CUSTOM` y usá `replace(inference_preset_config(...), ...)`. El resumen previo al workflow mostrará requisitos y escrituras sin exponer rutas privadas. Consultá la [guía central de presets](../../../docs/operations/notebook-configuration.md).

</details>


## 1. Preparar el entorno y elegir el flujo

El setup instala los paquetes y, a continuación, la única celda editable resuelve un preset tipado. Las demás celdas leen exclusivamente `WORKFLOW_CONFIG`.


In [ ]:
# Preparación del entorno: ejecutá esta celda una vez por runtime.
import importlib.util
import os
import runpy
import subprocess
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
WORKSPACE_DIR = Path("/content/vaaet")
if IN_COLAB:
    if (WORKSPACE_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(WORKSPACE_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(WORKSPACE_DIR)])
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    WORKSPACE_DIR = next(
        (
            path
            for path in candidates
            if (path / "vaaet-core/pyproject.toml").is_file()
            and (path / "vaaet-persistence/pyproject.toml").is_file()
            and (path / "vaaet-ml/pyproject.toml").is_file()
        ),
        None,
    )
    if WORKSPACE_DIR is None:
        raise RuntimeError("No se encontró el workspace VAAET con core y ML.")
CORE_ROOT = WORKSPACE_DIR / "vaaet-core"
PERSISTENCE_ROOT = WORKSPACE_DIR / "vaaet-persistence"
ML_ROOT = WORKSPACE_DIR / "vaaet-ml"
REPO_ROOT = ML_ROOT
os.chdir(ML_ROOT)
BOOTSTRAP = runpy.run_path(str(ML_ROOT / "scripts" / "notebook_bootstrap.py"))
RUNTIME = BOOTSTRAP["bootstrap_notebook"](
    workspace_root=WORKSPACE_DIR,
    core_root=CORE_ROOT,
    persistence_root=PERSISTENCE_ROOT,
    ml_root=ML_ROOT,
    core_extras=('vision', 'inference'),
    ml_extras=('visualization', 'database'),
    in_colab=IN_COLAB,
    framework='tensorflow',
    require_gpu=True,
)
VAAET_PACKAGE_FILE = RUNTIME.package_file
VAAET_ML_PACKAGE_FILE = RUNTIME.ml_package_file
GIT_COMMIT = RUNTIME.git_commit


In [ ]:
# Configuración del workflow: editá únicamente esta celda.
from dataclasses import replace

from vaaet_ml.workflow_presets import (
    InferencePreset,
    inference_preset_config,
    render_workflow_summary,
    resolve_inference_config,
)

SELECTED_PRESET = InferencePreset.PILOT_OFFLINE
CUSTOM_CONFIG = None
WORKFLOW_CONFIG = resolve_inference_config(
    SELECTED_PRESET,
    custom_config=CUSTOM_CONFIG,
)

print(render_workflow_summary(SELECTED_PRESET, WORKFLOW_CONFIG))


### Cómo elegir

Los presets incorporan las recetas soportadas y sus defaults seguros. Para cambiar una sola opción, partí de un preset conocido con `dataclasses.replace()`; no copies listas de variables globales. La configuración de Secrets y TLS está en la [guía de Colab](../../../docs/operations/colab-guide.md#secrets-y-postgresql).


In [ ]:
# Imports del workflow: no edites esta celda.
import os
import shutil

import cv2
import joblib
import numpy as np
import pandas as pd
import psycopg2
import sqlalchemy
import tensorflow as tf
import ultralytics

from vaaet.artifacts import MANIFEST_FILE, REQUIRED_FILES
from vaaet_persistence import PipelineRunMetadata, PipelineWorkflow, database_engine, inspect_database, persist_classified_telemetry, pipeline_run, prepare_persistence_run_metadata
from vaaet_ml.data.database import DatabaseProfile, get_optional_database_settings, load_reviewer_id
from vaaet_ml.data.review import HITL_CATALOG_FILE, HitlCatalogPublisher, HitlReviewCatalog, build_review_widget, finalize_review_session, load_review_queue, persist_human_validation, prepare_inference_review, recover_pending_review_validation, select_review_queue, sync_finalized_review_session
from vaaet_ml.evaluation.reporting import format_inference_result_summary, show_inference_dashboard
from vaaet.inference import TrafficStateEngine, load_traffic_bundle
from vaaet.inference.policy import empty_classification_result
from vaaet.inference.traffic_state import classify_raw_telemetry
from vaaet.logging import configure_logging
from vaaet_ml.notebook_io import resolve_video_input
from vaaet_ml.view_plan import load_video_view_plan
from vaaet_ml.settings import DRIVE_ARTIFACT_DIR, FEATURE_COLS, LABEL_MAP_PATH, MODEL_DIR, MODEL_PATH, RANDOM_SEED, SCALER_PATH, STATE_LABELS
from vaaet_ml.workflow_state import InferenceExecutionState, local_stage_attempt, record_stage_not_requested
from vaaet.telemetry import CANONICAL_RAW_TELEMETRY_COLUMNS
from vaaet.vision.analysis import analyze_video
from vaaet.vision.hud import HudConfig

configure_logging()
VIEW_PLAN = load_video_view_plan(WORKFLOW_CONFIG.view_plan_path)


In [ ]:
from importlib.metadata import version as package_version

_model_dir_abs = REPO_ROOT / MODEL_DIR
_model_dir_abs.mkdir(parents=True, exist_ok=True)
_ARTIFACT_NAMES = [*REQUIRED_FILES, MANIFEST_FILE]

def _bundle_paths(directory: Path) -> dict[str, Path]:
    return {name: directory / name for name in _ARTIFACT_NAMES}

paths = _bundle_paths(_model_dir_abs)
source = "local"
if not all(path.is_file() for path in paths.values()) and IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        drive_paths = _bundle_paths(Path("/content/drive") / DRIVE_ARTIFACT_DIR)
        if all(path.is_file() for path in drive_paths.values()):
            for name, drive_path in drive_paths.items():
                shutil.copy2(drive_path, paths[name])
            source = "Google Drive"
    except (ImportError, OSError) as error:
        print(f"⚠️ Google Drive no está disponible: {type(error).__name__}")


In [ ]:
if not all(path.is_file() for path in paths.values()) and IN_COLAB:
    from google.colab import files

    print("📤 Subí los cuatro archivos completos del bundle:", _ARTIFACT_NAMES)
    for name, content in files.upload().items():
        if name in _ARTIFACT_NAMES:
            paths[name].write_bytes(content)
    source = "upload"

missing = [name for name, path in paths.items() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"El bundle está incompleto; faltan: {missing}")
bundle = load_traffic_bundle(
    _model_dir_abs,
    allow_pilot=WORKFLOW_CONFIG.allow_pilot_bundle,
    allow_experimental=WORKFLOW_CONFIG.allow_experimental_bundle,
    persist_to_database=WORKFLOW_CONFIG.persist_to_database,
)
traffic_engine = TrafficStateEngine(bundle)
manifest, model, scaler = bundle.manifest, bundle.model, bundle.scaler
label_mapping = bundle.label_mapping
DEPLOYMENT_STAGE, MODEL_INPUT_POLICY = bundle.deployment_stage, bundle.input_policy
print(f"✅ Bundle {DEPLOYMENT_STAGE.upper()} válido, cargado desde {source} | input_policy={MODEL_INPUT_POLICY}")


## 3. Seleccionar el video

En Colab se solicita un MP4 mediante upload. En desarrollo local se intenta usar `data/sample/sample.mp4`. El nombre recomendado es `bridge_YYYY-MM-DD_HH-MM-SS_to_HH-MM-SS.mp4` para conservar trazabilidad temporal.

In [ ]:
VIDEO_PATH: Path | None = None  # Podés indicar un MP4 ya cargado para reutilizarlo.
_video_uploader = None
if IN_COLAB:
    from google.colab import files

    _video_uploader = files.upload
VIDEO_PATH = resolve_video_input(
    VIDEO_PATH,
    in_colab=IN_COLAB,
    uploader=_video_uploader,
    staging_directory=Path('/content') if IN_COLAB else REPO_ROOT / 'data/raw',
    local_fallback=REPO_ROOT / 'data/sample/sample.mp4',
)

print(f"Video seleccionado: {VIDEO_PATH or 'ninguno'}")
print("➡️ Siguiente paso: analizá el video." if VIDEO_PATH else "⚠️ Subí o seleccioná un MP4 para continuar.")

## 4. Procesar y clasificar el video

El video se procesa completo y el contrato produce una fila raw por cada ventana completa de 60 segundos. El primer minuto es la línea base de las features temporales: se necesitan dos minutos consecutivos para obtener la primera fila clasificable. La clasificación final ya se ejecuta en esta celda; no es necesario volver a calcularla después.

In [ ]:
analysis_result = None
df_telemetry = pd.DataFrame(columns=CANONICAL_RAW_TELEMETRY_COLUMNS)
df_classified = empty_classification_result(pd.DataFrame())
INFERENCE_PIPELINE_RUN_ID = None
LOCAL_INFERENCE_RUN_ID = None
REVIEW_VALIDATIONS = []
REVIEW_PENDING_VALIDATIONS = []
REVIEW_EXPORT_FRAME = None
INFERENCE_STATE = None
_review_session = None
for _callback in ('finalize_current_review', 'publish_pending_review', 'recover_pending_validation'):
    globals().pop(_callback, None)

if VIDEO_PATH is None or not VIDEO_PATH.is_file():
    raise FileNotFoundError("Seleccioná o subí un MP4 válido antes de continuar.")

OUTPUT_VIDEO = Path("/content") / f"{VIDEO_PATH.stem}_vaaet_analyzed.mp4" if IN_COLAB else VIDEO_PATH.with_name(f"{VIDEO_PATH.stem}_vaaet_analyzed.mp4")
_run_metadata = PipelineRunMetadata(workflow=PipelineWorkflow.INFERENCE, application_name="vaaet-ml-inference", application_version=package_version("vaaet-ml"), git_commit=GIT_COMMIT, source_kind="video", clip_id=VIDEO_PATH.stem, model_version=manifest["model_version"], model_revision=bundle.model_revision)
with pipeline_run(_run_metadata, local_manifest_directory=REPO_ROOT / "data/processed/pipeline-runs") as _run:
    analysis_result = analyze_video(
        VIDEO_PATH,
        OUTPUT_VIDEO,
        prediction_provider=traffic_engine.predict_latest,
        hud_config=HudConfig(debug=WORKFLOW_CONFIG.hud_debug),
        view_plan=VIEW_PLAN,
    )
    _run.set_output_rows(len(analysis_result.telemetry))
LOCAL_INFERENCE_RUN_ID = str(_run.id)
df_telemetry = analysis_result.telemetry
print(f'✅ Análisis nuevo completado | run={LOCAL_INFERENCE_RUN_ID}')


In [ ]:
for _callback in ('finalize_current_review', 'publish_pending_review', 'recover_pending_validation'):
    globals().pop(_callback, None)
REVIEW_VALIDATIONS = []
REVIEW_PENDING_VALIDATIONS = []
REVIEW_EXPORT_FRAME = None
INFERENCE_PIPELINE_RUN_ID = None
if analysis_result is None or LOCAL_INFERENCE_RUN_ID is None:
    raise RuntimeError("Primero ejecutá un análisis nuevo del video.")
INFERENCE_STATE = InferenceExecutionState(
    pipeline_run_id=LOCAL_INFERENCE_RUN_ID,
    model_revision=bundle.model_revision,
    classified=empty_classification_result(df_telemetry),
)
df_classified = INFERENCE_STATE.classified
try:
    if df_telemetry.empty:
        df_classified = INFERENCE_STATE.publish_insufficient_context()
        print(f"ℹ️ El video se procesó bien, pero no contiene un minuto completo.\n   Se omiten features, clasificación, PostgreSQL y revisión HITL.\n   Duración procesada: {analysis_result.processed_duration_seconds:.1f}s | mínimo: 60.0s")
    else:
        _classification_candidate = classify_raw_telemetry(
            df_telemetry,
            model,
            scaler,
            label_mapping=label_mapping,
            feature_cols=FEATURE_COLS,
            model_version=manifest["model_version"],
            model_revision=bundle.model_revision,
            input_policy=MODEL_INPUT_POLICY,
            inference_mode="stable",
            decision_policy=manifest["decision_policy"],
        )
        df_classified = INFERENCE_STATE.publish(
            _classification_candidate, analysis_result.classifications
        )
        if df_classified.empty:
            print("ℹ️ Hay telemetría, pero todavía falta contexto para un estado estable.\n   Se necesitan dos ventanas consecutivas de 60 segundos.\n   Se omiten métricas, PostgreSQL, revisión HITL y dashboard.")
        else:
            display(df_classified)
            print(format_inference_result_summary(df_classified, state_labels=STATE_LABELS))
        print(f"✅ Minutos de telemetría: {analysis_result.complete_minutes} | minutos clasificables: {len(df_classified)} | tramo descartado: {analysis_result.discarded_partial_seconds:.1f}s")
except Exception:
    INFERENCE_STATE.fail()
    df_classified = INFERENCE_STATE.classified
    raise


In [ ]:
print(f"✅ Video anotado: {analysis_result.video_path}")
print("➡️ Siguiente paso: revisá PostgreSQL y HITL según la configuración elegida.")
if IN_COLAB and WORKFLOW_CONFIG.download_annotated_video:
    from google.colab import files
    files.download(str(analysis_result.video_path))
elif IN_COLAB:
    print("ℹ️ Descarga desactivada; el video queda en /content.")

### Resultado de la clasificación

La celda anterior ya calculó `df_classified` mediante la cadena compartida `features → scaler → MLP → calibración → política temporal → detector de posible incidente`. No vuelvas a clasificar manualmente el mismo clip.

`traffic_state` sólo puede ser `Normal`, `Reduced` o `Congested`. Un posible accidente conserva `Congested` y activa `accident_rule_triggered`; `Accident` únicamente puede surgir de una validación humana.

## 5. Guardar en PostgreSQL (opcional)

Al activarla, el perfil `inference` escribe únicamente features y predicciones. Si la conexión, migración o permisos fallan, `df_classified` permanece disponible y el video no se pierde.

In [ ]:
INFERENCE_PIPELINE_RUN_ID = None
_persistence_manifest_dir = REPO_ROOT / "data/processed/pipeline-runs/persistence"
if WORKFLOW_CONFIG.persist_to_database:
    if INFERENCE_STATE is None or not INFERENCE_STATE.can_persist:
        print("ℹ️ PostgreSQL omitido: el intento vigente no tiene minutos clasificados verificables.")
    else:
        INFERENCE_STATE.begin_persistence()
        try:
            with local_stage_attempt(
                _persistence_manifest_dir, pipeline_run_id=LOCAL_INFERENCE_RUN_ID, stage="postgresql-persistence"
            ) as _persistence_attempt:
                inference_settings = get_optional_database_settings(DatabaseProfile.INFERENCE)
                if inference_settings is None:
                    raise RuntimeError("La persistencia PostgreSQL solicitada no puede comenzar: falta el perfil inference.")
                with database_engine(inference_settings) as db_engine:
                    health = inspect_database(db_engine, DatabaseProfile.INFERENCE)
                    print(f"PostgreSQL {health.server_version} | TLS={health.ssl_enabled} | schemas={health.available_schemas}")
                    _db_metadata = prepare_persistence_run_metadata(INFERENCE_STATE.classified, workflow=PipelineWorkflow.INFERENCE, application_name="vaaet-ml-inference", application_version=package_version("vaaet-ml"), git_commit=GIT_COMMIT, model_version=manifest["model_version"], model_revision=bundle.model_revision)
                    with pipeline_run(_db_metadata, engine=db_engine, run_id=LOCAL_INFERENCE_RUN_ID) as _db_run:
                        persisted = persist_classified_telemetry(
                            INFERENCE_STATE.classified, engine=db_engine, model_version=manifest["model_version"],
                            model_revision=bundle.model_revision, pipeline_run_id=_db_run.id,
                        )
                        _db_run.set_output_rows(len(INFERENCE_STATE.classified))
                _persistence_attempt.set_output_rows(len(INFERENCE_STATE.classified))
            if _db_run.outcome is None or not _db_run.outcome.work_succeeded:
                raise RuntimeError("PostgreSQL did not publish a confirmed persistence outcome.")
            _audit_complete = _persistence_attempt.audit_complete and _db_run.outcome.audit_complete
            INFERENCE_STATE.complete_persistence(audit_complete=_audit_complete)
            INFERENCE_PIPELINE_RUN_ID = LOCAL_INFERENCE_RUN_ID
            print(f"✅ PostgreSQL: {persisted.telemetry_rows} filas procesadas | nuevas: {persisted.inserted_telemetry_rows} features + {persisted.inserted_classification_rows} predicciones | run={INFERENCE_PIPELINE_RUN_ID}")
            if not INFERENCE_STATE.persistence_audit_complete:
                print("⚠️ Los datos fueron guardados, pero la auditoría local o PostgreSQL quedó incompleta; la revisión permanece bloqueada.")
                print("   Recuperación: verificá estas mismas filas con reconcile_classified_telemetry(); esa operación no vuelve a insertarlas.")
        except Exception as error:
            INFERENCE_STATE.fail_persistence()
            print(f"🔴 Falló PostgreSQL ({type(error).__name__}). Revisá migración, TLS y permisos del rol.")
            print("   El video y la clasificación se conservan, pero la revisión PostgreSQL queda bloqueada.")
            raise RuntimeError("La persistencia PostgreSQL solicitada no terminó.") from None
else:
    if LOCAL_INFERENCE_RUN_ID is not None:
        record_stage_not_requested(
            _persistence_manifest_dir, pipeline_run_id=LOCAL_INFERENCE_RUN_ID, stage="postgresql-persistence"
        )
    print("ℹ️ PostgreSQL desactivado por la configuración central.")
    print("   Las filas siguen en df_classified y pueden entrar en una revisión HITL portable.")
print("➡️ Siguiente paso: iniciá la revisión humana si la habilitaste.")


## 6. Revisión humana HITL

La revisión se abre **después** de terminar el clip; no interrumpe la inferencia con pop-ups. Mirá el video anotado, buscá el `record_time` indicado y contrastá también los minutos anterior y posterior.

- **Confirmar:** conservar el estado predicho.
- **Corregir:** elegir `Normal`, `Reduced` o `Congested`.
- **Omitir:** la fila queda pendiente y nunca se usa como ground truth.
- **Confirmar Accident:** elegir `Accident`, marcar `I reviewed temporal context` y escribir una nota. El estado automático permanece `Congested`; Accident sólo existe como decisión humana.

Con PostgreSQL, `Save validation` inserta una decisión append-only en `vaaet_feedback.human_validations`; nunca edites la predicción manualmente. Si ves **Guardado; auditoría pendiente**, el formulario no avanza ni la decisión se exporta: repetí el mismo botón o usá `recover_pending_validation("<validation-id>")` mientras siga abierta **la misma sesión**. Esa recuperación verifica la decisión original y nunca vuelve a insertarla. Si reiniciaste Colab o cambiaste de clip, primero tenés que reconstruir explícitamente la sesión de la corrida original con sus predicciones y features; no recuperes la decisión dentro del clip nuevo. Sin PostgreSQL, las decisiones quedan en el paquete portable. Al terminar ejecutá `finalize_current_review()` para sellar el ZIP local. Ese paso no publica el catálogo: sólo el runtime designado como publicador debe ejecutar después `publish_pending_review(...)`. El reentrenamiento se realiza exclusivamente en `train_traffic_state_classifier.ipynb`.


In [ ]:
_review_has_rows = INFERENCE_STATE is not None and INFERENCE_STATE.can_persist
_review_storage_ready = INFERENCE_STATE is not None and INFERENCE_STATE.can_review(require_persistence=WORKFLOW_CONFIG.persist_to_database)
_review_enabled = WORKFLOW_CONFIG.enable_human_review and _review_storage_ready
if WORKFLOW_CONFIG.enable_human_review and not _review_has_rows:
    print('ℹ️ Revisión HITL omitida: no hay minutos clasificables; no se cargó VAAET_REVIEWER_ID.')
if WORKFLOW_CONFIG.enable_human_review and _review_has_rows and not _review_storage_ready:
    raise RuntimeError('La revisión PostgreSQL está bloqueada porque la persistencia de inferencia no terminó.')
review_settings = None
if _review_enabled and WORKFLOW_CONFIG.persist_to_database:
    review_settings = get_optional_database_settings(DatabaseProfile.REVIEW)
    if review_settings is None:
        raise RuntimeError('La revisión PostgreSQL está habilitada, pero falta el perfil review.')
_review_attempt_id = INFERENCE_STATE.attempt_id if INFERENCE_STATE is not None else None
def _review_is_current(attempt_id=_review_attempt_id):
    return INFERENCE_STATE is not None and attempt_id is not None and INFERENCE_STATE.attempt_id == attempt_id and INFERENCE_STATE.can_review(require_persistence=WORKFLOW_CONFIG.persist_to_database)
_review_session = prepare_inference_review(
    enabled=_review_enabled,
    classified=df_classified,
    inference_pipeline_run_id=INFERENCE_PIPELINE_RUN_ID,
    reviewer_id=load_reviewer_id() if _review_enabled else None,
    settings=review_settings,
    mode=WORKFLOW_CONFIG.review_mode,
    is_current=_review_is_current,
)
REVIEW_VALIDATIONS = _review_session.validations
REVIEW_PENDING_VALIDATIONS = _review_session.pending_validations
REVIEW_EXPORT_FRAME = _review_session.export_frame


In [ ]:
def recover_pending_validation(validation_id, attempt_id=_review_attempt_id, session=_review_session):
    """Reconcilia una decisión PostgreSQL existente sin repetir su escritura."""
    INFERENCE_STATE.require_attempt(attempt_id)
    settings = get_optional_database_settings(DatabaseProfile.REVIEW)
    if settings is None:
        raise RuntimeError('Falta el perfil PostgreSQL review para recuperar la auditoría.')
    result = recover_pending_review_validation(
        session, validation_id, settings=settings
    )
    if result.confirmed:
        print(f'✅ Validación {result.decision.validation_id} confirmada para la sesión original.')
    else:
        print(f'⚠️ Validación {result.decision.validation_id} guardada; auditoría aún pendiente.')
    return result


In [ ]:
if REVIEW_EXPORT_FRAME is not None and INFERENCE_STATE is not None:
    _finalization_attempt_id = INFERENCE_STATE.attempt_id
    def finalize_current_review(attempt_id=_finalization_attempt_id, session=_review_session, export_frame=REVIEW_EXPORT_FRAME, validations=REVIEW_VALIDATIONS, pending_validations=REVIEW_PENDING_VALIDATIONS):
        """Sella localmente la revisión vigente sin publicar el catálogo."""
        INFERENCE_STATE.require_attempt(attempt_id)
        if not INFERENCE_STATE.can_review(require_persistence=WORKFLOW_CONFIG.persist_to_database):
            raise RuntimeError("La revisión ya no pertenece a un intento vigente y trazable.")
        result = finalize_review_session(
            classified=export_frame,
            validations=validations,
            pending_validations=pending_validations,
            session=session,
            pipeline_run_id=INFERENCE_PIPELINE_RUN_ID or LOCAL_INFERENCE_RUN_ID,
            model_version=manifest["model_version"],
            git_commit=GIT_COMMIT,
            vaaet_version=package_version("vaaet-ml"),
            local_root=REPO_ROOT / "data/processed/hitl-reviews",
            canonical_root=None,
        )
        print(f"📦 HITL package {result.package_id} | reviewed={result.reviewed_rows} | pending={result.pending_rows}\n   fingerprint={result.fingerprint} | status={result.sync_status}\n   location={result.local_path}")
        print("ℹ️ ZIP local sellado. Publicalo sólo desde el runtime designado con publish_pending_review(result.local_path).")
        return result
    def publish_pending_review(local_path):
        """Publica un ZIP sellado desde el único runtime coordinador."""
        pending = sync_finalized_review_session(local_path, canonical_root=Path('.'), publisher=None)
        if not IN_COLAB:
            print("ℹ️ Drive no está disponible fuera de Colab; el ZIP conserva estado pending-sync.")
            return pending
        try:
            from google.colab import drive  # type: ignore[import-untyped]
            drive.mount("/content/drive", force_remount=False)
        except Exception as sync_error:
            print(f"⚠️ Drive no está disponible ({type(sync_error).__name__}); el ZIP local queda pending-sync.")
            return pending
        canonical_root = Path("/content/drive/MyDrive/vaaet-ml/data/hitl-reviews")
        catalog = HitlReviewCatalog(canonical_root / HITL_CATALOG_FILE)
        lock_root = REPO_ROOT / "data/processed/hitl-publisher-locks"
        with HitlCatalogPublisher(catalog, lock_directory=lock_root) as publisher:
            result = sync_finalized_review_session(
                local_path, canonical_root=canonical_root, publisher=publisher
            )
        print(f"📤 HITL package {result.package_id} | status={result.sync_status} | location={result.canonical_path or result.local_path}")
        if result.sync_error:
            print(f"⚠️ El ZIP local sigue pendiente: {result.sync_error}")
        return result
    print("Después de revisar u omitir filas, ejecutá finalize_current_review() una vez. Repetirlo es idempotente.")
    print("El runtime coordinador puede publicar luego con publish_pending_review(result.local_path).")


## 7. Ver el resumen visual

El dashboard resume distribución de estados, velocidad, confianza, tipos de vehículos y relación velocidad/volumen. Sólo se ejecuta cuando el dashboard habilitado por el preset y existen minutos clasificados.

In [ ]:
if not WORKFLOW_CONFIG.show_dashboard:
    print("ℹ️ Dashboard desactivado por la configuración central.")
elif 'df_classified' not in globals():
    print("⚠️ Primero ejecutá el análisis del video.")
elif INFERENCE_STATE is None or not INFERENCE_STATE.can_persist:
    print("ℹ️ No hay minutos clasificables para mostrar. Se necesitan dos minutos completos consecutivos.")
else:
    show_inference_dashboard(INFERENCE_STATE.classified, state_labels=label_mapping)
    print("✅ Dashboard generado.")
    print("➡️ Fin del flujo: conservá el video y, si revisaste filas, finalizá el paquete HITL.")